# Phase 3: Reproduce & Analyze KnowGPT (RL-based Retrieval)

**Paper:** KnowGPT: Knowledge Graph based Prompting for LLMs (NeurIPS 2024)

**Muc tieu:**
1. Hieu ro RL pipeline cho graph retrieval (da run o KnowGPT_v3.ipynb)
2. Document lai MDP formulation, policy network, reward
3. Chuyen sang WebQSP de so sanh voi GNN-RAG
4. Rut ra bai hoc cho proposed method

**KnowGPT pipeline:**
```
Question -> Entity Extraction -> Subgraph Retrieval (ConceptNet)
    -> Deep RL (Policy Network) -> Knowledge Paths
    -> MAB Prompt Construction -> LLM Answer
```

**Ly thuyet RL can nam truoc khi doc phase nay:**
- MDP: (S, A, P, R, gamma)
- Policy Gradient / REINFORCE
- Reward shaping
- Ref: Sutton & Barto Ch.3,4,13

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/ThS/Research/colab/graph_rag')
from common import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
from tqdm import tqdm

## 1. MDP Formulation cua KnowGPT

### Review tu paper va code da chay (KnowGPT_v3.ipynb):

| Component | KnowGPT |
|-----------|----------|
| **State** | (current_node, context_encoding) |
| **Action** | Chon edge/neighbor tiep theo |
| **Reward** | Reachability + Context-relatedness + Conciseness |
| **Policy** | MLP policy network, train bang Policy Gradient |
| **Episode** | Bat dau tu source entity, ket thuc khi dat target hoac max steps |

### Nhan xet:
- KnowGPT dung RL cho **path finding** (source -> target)
- Reward = mix cua 3 yeu to: reach target + lien quan + ngan gon
- Khong co **explicit budget control** (chi co conciseness penalty)
- Khong dung GNN features lam heuristic
- => Proposed method se bo sung: GNN heuristic + budget constraint + better reward

## 2. Implement RL Components (simplified version)

Re-implement cac component chinh de hieu ro truoc khi modify.

In [ ]:
class KGEnvironment:
    """Knowledge Graph environment cho RL agent.
    
    Moi episode: agent bat dau tu 1 entity, di chuyen tren graph
    de thu thap tri thuc lien quan den question.
    """
    
    def __init__(self, graph, entity2id, relation2id, 
                 entity_embeddings=None, max_steps=5):
        self.graph = graph  # NetworkX graph
        self.entity2id = entity2id
        self.relation2id = relation2id
        self.entity_embs = entity_embeddings
        self.max_steps = max_steps
        
        # Episode state
        self.current_node = None
        self.visited_path = []
        self.step_count = 0
        self.collected_triples = []
    
    def reset(self, start_entity):
        """Reset environment cho episode moi."""
        self.current_node = start_entity
        self.visited_path = [start_entity]
        self.step_count = 0
        self.collected_triples = []
        return self._get_state()
    
    def _get_state(self):
        """Tra ve state hien tai."""
        return {
            'current_node': self.current_node,
            'visited_path': list(self.visited_path),
            'step_count': self.step_count,
            'available_actions': self._get_actions(),
        }
    
    def _get_actions(self):
        """Lay danh sach actions kha thi (neighbors chua tham)."""
        if self.current_node not in self.graph:
            return ['STOP']
        
        neighbors = []
        for n in self.graph.successors(self.current_node):
            if n not in self.visited_path:  # Khong quay lai
                for edge_data in self.graph[self.current_node][n].values():
                    rel = edge_data.get('relation', 'related')
                    neighbors.append((rel, n))
        
        # Luon co option STOP
        actions = neighbors + [('STOP', None)]
        return actions
    
    def step(self, action_idx):
        """Thuc hien action, tra ve (next_state, reward, done)."""
        actions = self._get_actions()
        
        if action_idx >= len(actions):
            action_idx = len(actions) - 1  # Default to STOP
        
        relation, next_node = actions[action_idx]
        
        if relation == 'STOP' or self.step_count >= self.max_steps:
            done = True
            reward = 0  # Terminal reward se duoc tinh rieng
        else:
            # Di chuyen den node moi
            self.collected_triples.append(
                (self.current_node, relation, next_node)
            )
            self.current_node = next_node
            self.visited_path.append(next_node)
            self.step_count += 1
            done = (self.step_count >= self.max_steps or 
                    len(self._get_actions()) <= 1)  # Chi con STOP
            reward = 0  # Intermediate reward = 0 (sparse reward)
        
        return self._get_state(), reward, done


print('KGEnvironment defined.')

In [ ]:
class PolicyNetwork(nn.Module):
    """Policy network cho RL retrieval agent.
    
    Input: [query_emb; current_node_emb; history_emb]
    Output: probability distribution tren actions
    
    Tuong tu KnowGPT nhung don gian hoa.
    """
    
    def __init__(self, emb_dim=256, hidden_dim=256, max_actions=50):
        super().__init__()
        # Input: query_emb + current_node_emb + history_emb
        input_dim = emb_dim * 3
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Score moi action bang dot product voi action embedding
        self.action_scorer = nn.Linear(hidden_dim, emb_dim)
    
    def forward(self, query_emb, node_emb, history_emb, action_embs):
        """Tinh probability distribution tren actions.
        
        Args:
            query_emb: [batch, emb_dim]
            node_emb: [batch, emb_dim]
            history_emb: [batch, emb_dim]
            action_embs: [batch, num_actions, emb_dim]
        
        Returns:
            action_probs: [batch, num_actions]
        """
        # Concat inputs
        state = torch.cat([query_emb, node_emb, history_emb], dim=-1)
        
        # Encode state
        hidden = self.network(state)  # [batch, hidden_dim]
        state_repr = self.action_scorer(hidden)  # [batch, emb_dim]
        
        # Score actions bang dot product
        scores = torch.bmm(action_embs, 
                           state_repr.unsqueeze(-1)).squeeze(-1)  # [batch, num_actions]
        
        # Softmax -> probabilities
        action_probs = F.softmax(scores, dim=-1)
        return action_probs


print('PolicyNetwork defined.')
# Test
policy = PolicyNetwork(emb_dim=128, hidden_dim=128)
print(f'Parameters: {sum(p.numel() for p in policy.parameters()):,}')

## 3. Reward Function

### KnowGPT Reward:
- **Reachability**: +1 neu dat duoc target entity
- **Context-relatedness**: cosine similarity giua path va question
- **Conciseness**: penalty cho path dai

### Cai tien cho proposed method (Phase 4):
- **Answer quality**: F1/EM cua LLM answer (end-to-end reward)
- **Budget penalty**: penalty khi vuot qua budget
- **Diversity bonus**: khuyen khich explore paths khac nhau

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('all-MiniLM-L6-v2')


def compute_reward_knowgpt(question, collected_triples, target_entities,
                            alpha=0.4, beta=0.4, gamma=0.2, max_path_len=5):
    """Reward function theo phong cach KnowGPT.
    
    R = alpha * reachability + beta * relevance + gamma * conciseness
    """
    if not collected_triples:
        return 0.0
    
    # 1. Reachability: dat duoc target entity?
    visited = set()
    for h, r, t in collected_triples:
        visited.add(h)
        visited.add(t)
    reachability = float(any(e in visited for e in target_entities))
    
    # 2. Context-relatedness: path co lien quan den question?
    path_text = verbalize_triples(collected_triples)
    q_emb = encoder.encode([question])
    p_emb = encoder.encode([path_text])
    relevance = float(np.dot(q_emb[0], p_emb[0]) / 
                      (np.linalg.norm(q_emb[0]) * np.linalg.norm(p_emb[0]) + 1e-8))
    relevance = max(0, relevance)  # Clip negative
    
    # 3. Conciseness: path cang ngan cang tot
    conciseness = max(0, 1 - len(collected_triples) / max_path_len)
    
    reward = alpha * reachability + beta * relevance + gamma * conciseness
    return reward


# Test
test_reward = compute_reward_knowgpt(
    question='Who directed Inception?',
    collected_triples=[('Inception', 'directed_by', 'Christopher Nolan')],
    target_entities=['Christopher Nolan']
)
print(f'Test reward: {test_reward:.4f}')

## 4. REINFORCE Training Loop

**Algorithm:**
```
for episode in episodes:
    1. Sample trajectory tau = (s0, a0, r0, s1, a1, r1, ...)
    2. Compute return G_t = sum of future rewards
    3. Update policy: theta += lr * G_t * grad(log pi(a_t|s_t))
```

**Ref:** Sutton & Barto, Chapter 13 (Policy Gradient Methods)

In [ ]:
def reinforce_train_step(policy, optimizer, env, question, query_emb,
                          entity_embs_dict, target_entities, gamma=0.99):
    """1 training step cua REINFORCE.
    
    Day la simplified version de hieu algorithm.
    Phase 4 se implement day du hon.
    """
    emb_dim = query_emb.shape[-1]
    
    # Collect trajectory
    log_probs = []
    states = []
    state = env.reset(list(entity_embs_dict.keys())[0])  # Start tu first entity
    
    done = False
    while not done:
        actions = state['available_actions']
        if len(actions) <= 1:  # Chi co STOP
            break
        
        # Encode state
        current = state['current_node']
        node_emb = entity_embs_dict.get(current, torch.zeros(emb_dim))
        
        # History embedding: mean cua visited nodes
        history_embs = [entity_embs_dict.get(n, torch.zeros(emb_dim)) 
                        for n in state['visited_path']]
        history_emb = torch.stack(history_embs).mean(dim=0)
        
        # Action embeddings
        action_emb_list = []
        for rel, node in actions:
            if node is not None:
                action_emb_list.append(entity_embs_dict.get(node, torch.zeros(emb_dim)))
            else:  # STOP action
                action_emb_list.append(torch.zeros(emb_dim))
        action_embs = torch.stack(action_emb_list).unsqueeze(0)
        
        # Forward policy
        q = query_emb.unsqueeze(0)
        n = node_emb.unsqueeze(0)
        h = history_emb.unsqueeze(0)
        
        probs = policy(q, n, h, action_embs)  # [1, num_actions]
        
        # Sample action
        dist = torch.distributions.Categorical(probs)
        action_idx = dist.sample()
        log_prob = dist.log_prob(action_idx)
        log_probs.append(log_prob)
        
        # Step environment
        state, reward, done = env.step(action_idx.item())
    
    # Compute terminal reward
    terminal_reward = compute_reward_knowgpt(
        question, env.collected_triples, target_entities
    )
    
    # REINFORCE loss: -sum(log_prob * reward)
    if log_probs:
        loss = -torch.stack(log_probs).sum() * terminal_reward
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        return loss.item(), terminal_reward
    
    return 0.0, terminal_reward


print('REINFORCE training function defined.')

## 5. Review lai KnowGPT_v3 results

Load va phan tich ket qua da chay o KnowGPT_v3.ipynb

In [ ]:
# === Load results tu KnowGPT_v3 ===
# TODO: Update paths cho dung

# EXTRACTED_TRAIN_PATH = '/content/drive/MyDrive/ABC/KnowGPT/extracted_train_paths_v3.pkl'
# MAB_TRAIN_PATH = '/content/drive/MyDrive/ABC/KnowGPT/mab_train_paths_v3.pkl'
# RESULT_TRAIN_PATH = '/content/drive/MyDrive/ABC/KnowGPT/mab_training_log_v3.csv'

# if os.path.exists(RESULT_TRAIN_PATH):
#     import pandas as pd
#     results_df = pd.read_csv(RESULT_TRAIN_PATH)
#     print(results_df.describe())
#     
#     # Plot training curve
#     import matplotlib.pyplot as plt
#     fig, axes = plt.subplots(1, 2, figsize=(12, 4))
#     results_df['reward'].rolling(100).mean().plot(ax=axes[0], title='Reward')
#     results_df['accuracy'].rolling(100).mean().plot(ax=axes[1], title='Accuracy')
#     plt.tight_layout()
#     plt.show()

print('Load va analyze KnowGPT results')
print('Compare voi: Phase 1 baselines va Phase 2 GNN-RAG')

## 6. So sanh KnowGPT vs GNN-RAG: Lessons Learned

| Aspect | GNN-RAG | KnowGPT | Proposed (Phase 4) |
|--------|---------|---------|--------------------|
| Retrieval | One-shot GNN scoring | Sequential RL | GNN heuristic + RL sequential |
| Path selection | Shortest path (fixed) | RL-learned path | RL-learned + greedy pruning |
| Budget control | No | Conciseness penalty | Explicit budget constraint |
| Heuristic | GNN scores | None | GNN scores as features |
| Stopping | No | No | Learned STOP action |
| Reward | N/A (supervised) | Reach + Relevance + Concise | End-to-end answer quality |

### Key insights:
1. GNN-RAG co GNN heuristic tot nhung one-shot
2. KnowGPT co RL sequential nhung khong co heuristic guidance
3. => **Ket hop**: dung GNN scores lam features cho RL policy -> best of both worlds

## Doc them truoc Phase 4

- [ ] MINERVA (ICLR 2018): MDP formulation goc cho KG reasoning
- [ ] DeepPath (EMNLP 2017): RL path finding, reward shaping
- [ ] Sutton & Barto Ch.13: Policy Gradient, REINFORCE with baseline
- [ ] Reward shaping (Ng et al. 1999): potential-based shaping de giam sparse reward